# Bronze Ingestion

This notebook implements a reusable, configuration-driven Bronze ingestion pipeline using Databricks Auto Loader.

It incrementally ingests dataset-specific CSV files from Amazon S3 through Unity Catalog external volumes and writes them to Bronze Delta tables.

Each execution processes a single dataset, preserving source lineage, ingestion metadata, and rescued data for downstream validation.

## Responsibilities

- Resolve dataset-specific configurations and schemas
- Read source files from Unity Catalog external volumes backed by Amazon S3
- Apply explicit source schemas
- Incrementally process files using Auto Loader
- Preserve source file metadata and ingestion audit columns
- Capture schema mismatches in `_rescued_data` for downstream validation
- Write source records to Bronze Delta tables
- Maintain independent checkpoints for each dataset to support incremental processing and recovery

In [0]:
import sys
import uuid

from pathlib import Path
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("platform_schema", "sbx_platform")
dbutils.widgets.text("bronze_schema", "sbx_bronze")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("dataset_name", "")  # Required; set explicitly per run

In [0]:
import uuid

catalog = dbutils.widgets.get("catalog").strip()
platform_schema = dbutils.widgets.get("platform_schema").strip()
bronze_schema = dbutils.widgets.get("bronze_schema").strip()
provided_run_id = dbutils.widgets.get("run_id").strip()
dataset_name = dbutils.widgets.get("dataset_name").strip()

required_params = {
    "catalog": catalog,
    "platform_schema": platform_schema,
    "bronze_schema": bronze_schema,
    "dataset_name": dataset_name,
}
for param_name, param_value in required_params.items():
    if not param_value:
        raise ValueError(f"Required parameter is missing: {param_name}")

run_id = provided_run_id or str(uuid.uuid4())
print(f"Dataset: {dataset_name} | Run ID: {run_id}")
print(f"Catalog: {catalog} | Platform: {platform_schema} | Bronze: {bronze_schema}")

In [0]:
import sys
from pathlib import Path

src_path = Path.cwd().parent / "src"
if not src_path.is_dir():
    raise FileNotFoundError(f"Project source directory not found: {src_path}")
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from dbx_platform.config import get_bronze_config

bronze_configs = get_bronze_config(
    catalog=catalog,
    platform_schema=platform_schema,
    bronze_schema=bronze_schema,
)
if dataset_name not in bronze_configs:
    raise ValueError(
        f"Unknown dataset: {dataset_name}. Available: {list(bronze_configs)}"
    )
selected_config = bronze_configs[dataset_name]

for key in ("source_path", "checkpoint_path", "target_table"):
    if not selected_config.get(key):
        raise ValueError(f"Missing {key} in configuration for {dataset_name}")

print(f"Source: {selected_config['source_path']}")
print(f"Checkpoint: {selected_config['checkpoint_path']}")
print(f"Target: {selected_config['target_table']}")

In [0]:
source_path = selected_config["source_path"]
try:
    source_entries = dbutils.fs.ls(source_path)
except Exception as exc:
    raise RuntimeError(
        f"Unable to access source directory for {dataset_name}: {source_path}"
    ) from exc

csv_files = [
    item for item in source_entries
    if item.name.lower().endswith(".csv") and item.size > 0
]
print(f"Source directory accessible: {source_path}")
print(f"Non-empty CSV files currently present: {len(csv_files)}")
if not csv_files:
    print("No CSV files currently present; continuing without a pre-ingestion failure.")

In [0]:
from dbx_platform.schemas import get_dataset_schema
from pyspark.sql.types import StructType

selected_schema = get_dataset_schema(dataset_name)
if not isinstance(selected_schema, StructType) or not selected_schema.fields:
    raise ValueError(f"Invalid or empty schema for dataset: {dataset_name}")

field_names = [field.name for field in selected_schema.fields]
if len(field_names) != len(set(field_names)):
    raise ValueError(f"Duplicate schema fields for dataset: {dataset_name}")

print(f"Schema validated: {dataset_name} ({len(field_names)} fields)")
spark.createDataFrame([], selected_schema).printSchema()

In [0]:
from dbx_platform.ingestion import build_bronze_stream

selected_stream = build_bronze_stream(
    spark=spark,
    source_path=selected_config["source_path"],
    schema=selected_schema,
    run_id=run_id,
)
print(f"Auto Loader stream initialised: {dataset_name}")
selected_stream.printSchema()

In [0]:

# Writes to the configured Delta table using the dataset-specific checkpoint. With availableNow=True, a run can succeed without new input files.


from dbx_platform.ingestion import execute_bronze_ingestion

target_table = selected_config["target_table"]
checkpoint_path = selected_config["checkpoint_path"]
query_name = f"bronze_{dataset_name}_ingestion"

print(f"Starting Bronze ingestion: {dataset_name}")
print(f"Target table: {target_table}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Run ID: {run_id}")

query = execute_bronze_ingestion(
    stream_df=selected_stream,
    target_table=target_table,
    checkpoint_path=checkpoint_path,
    query_name=query_name,
)
print(f"Bronze ingestion completed: {dataset_name}")
print(f"Query ID: {query.id}")
print(f"Status: {query.status}")


In [0]:
# Validates the whole existing Bronze table, not just rows written in this run. If old rows are missing audit metadata, validation fails even when the latest run succeeded.

from pyspark.sql import functions as F

target_table = selected_config["target_table"]
bronze_df = spark.table(target_table)
required_audit_columns = [
    "_source_file",
    "_source_modified_at",
    "_ingested_at",
    "_ingestion_run_id",
]
missing_columns = [col for col in required_audit_columns if col not in bronze_df.columns]
if missing_columns:
    raise ValueError(f"Missing audit columns in {target_table}: {missing_columns}")

invalid_audit_count = bronze_df.filter(
    F.col("_source_file").isNull()
    | F.col("_source_modified_at").isNull()
    | F.col("_ingested_at").isNull()
    | F.col("_ingestion_run_id").isNull()
).count()
if invalid_audit_count:
    raise ValueError(f"Found {invalid_audit_count} rows with missing audit metadata")

total_records = bronze_df.count()
current_run_records = bronze_df.filter(F.col("_ingestion_run_id") == run_id).count()
print(f"Dataset: {dataset_name}")
print(f"Target table: {target_table}")
print(f"Total Bronze records: {total_records}")
print(f"Records ingested in current run: {current_run_records}")
print(f"Invalid audit records: {invalid_audit_count}")
display(bronze_df.limit(10))
print("Bronze ingestion validation completed successfully.")

